# 🐼 Módulo 2 · Sesión 1
## Pandas: tu primer dataset

---

**Contexto:** La cafetería de Intenalco lleva el registro de sus ventas en un archivo CSV.  
Tiene 28 filas — una semana de datos. Puedes contarlas con los dedos.

**Objetivo de hoy:** hacerle 5 preguntas de negocio a ese dataset usando Pandas.  
Sin fórmulas. Sin tablas dinámicas. Sin copiar y pegar.

---

**Cómo usar este cuaderno**
- Cada celda gris tiene código. Ejecútala con **Shift + Enter** o el botón ▶
- Si algo no entiendes, selecciona el código y presiona el botón ✦ de Colab para pedirle explicación a la IA
- Los bloques con 💬 son sugerencias de prompts para el asistente IA

> **Regla del módulo:** no memorices sintaxis. Entiende qué hace cada operación y deja que la IA te ayude a escribirla.

---
## Parte 1 — Cargar los datos

Lo primero siempre es lo mismo: decirle a Pandas dónde están los datos.  
En este caso el archivo vive en GitHub — no necesitas descargarlo ni subirlo.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/IntenalcoFormacion/datasets/main/L1_cafeteria_intenalco.csv"

df = pd.read_csv(URL)

print("✓ Datos cargados correctamente")
print(f"  Filas:    {len(df)}")
print(f"  Columnas: {len(df.columns)}")
print(f"  Columnas: {list(df.columns)}")

> **¿Qué acaba de pasar?**  
> `pd.read_csv(URL)` descargó el archivo y lo convirtió en un **DataFrame** — que es exactamente una tabla: filas y columnas.  
> Lo guardamos en la variable `df`. A partir de ahora, `df` *es* la cafetería.

---
## Parte 2 — Ver los datos

Antes de cualquier análisis, hay que pararse en la esquina y mirar el campo completo.

In [ ]:
# Las primeras 5 filas — como hacer Ctrl+Inicio en Excel
df.head()

In [ ]:
# Resumen técnico: tipos de datos y si hay vacíos
df.info()

In [ ]:
# Estadísticas automáticas de las columnas numéricas
df.describe()

> **Qué mirar en `describe()`**  
> - `count`: cuántos registros tienen ese dato (si es menor que el total, hay vacíos)  
> - `mean`: el promedio  
> - `min` / `max`: los extremos  
> - `50%`: la mediana — el valor del medio si ordenas todos de menor a mayor
>
> 💬 **Prompt para la IA:** *"Explícame qué significa cada fila del resultado de describe() aplicado a este dataset de ventas de cafetería"*

---
## Parte 3 — Las 5 preguntas de negocio

Un análisis no empieza por el código — empieza por la pregunta.  
Estas son las 5 preguntas que le vamos a hacer a los datos de la cafetería.

### Pregunta 1 — ¿Cuánto vendió la cafetería en la semana?

In [ ]:
total_semana = df['total'].sum()

print(f"Total vendido en la semana: ${total_semana:,.0f}")

> `df['total']` selecciona la columna *total* — como hacer clic en el encabezado de una columna en Excel.  
> `.sum()` la suma. Así de simple.

### Pregunta 2 — ¿Cuál fue el día con más ventas?

In [ ]:
ventas_por_dia = df.groupby('dia')['total'].sum().sort_values(ascending=False)

print("Ventas por día (de mayor a menor):")
print(ventas_por_dia.to_string())

> `groupby('dia')` agrupa las filas por día — como arrastrar 'día' al área de filas de una tabla dinámica.  
> `['total'].sum()` suma la columna total dentro de cada grupo.  
> `.sort_values(ascending=False)` ordena de mayor a menor.
>
> 💬 **Prompt para la IA:** *"Modifica este código para que muestre también el número de productos vendidos por día, no solo el total en pesos"*

### Pregunta 3 — ¿Bebidas o comidas: cuál categoría genera más ingresos?

In [ ]:
por_categoria = df.groupby('categoria').agg(
    total_vendido = ('total',    'sum'),
    unidades      = ('unidades', 'sum'),
    num_productos = ('producto', 'nunique')
).reset_index()

por_categoria

> `.agg()` permite calcular varias cosas a la vez sobre el mismo grupo.  
> `'nunique'` cuenta cuántos valores distintos hay — en este caso, cuántos productos diferentes tiene cada categoría.

### Pregunta 4 — ¿Cuáles son los 3 productos más vendidos en pesos?

In [ ]:
top_productos = (
    df.groupby('producto')['total']
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .reset_index()
)

top_productos.columns = ['Producto', 'Total vendido']
top_productos['Total vendido'] = top_productos['Total vendido'].apply(lambda x: f"${x:,.0f}")
top_productos

> El paréntesis exterior permite encadenar varias operaciones en orden, de arriba a abajo.  
> Léelo como una receta: *agrupa por producto → suma el total → ordena → quédate con los 3 primeros*.
>
> 💬 **Prompt para la IA:** *"Modifica el código para mostrar los 3 productos con MÁS unidades vendidas, no los de mayor ingreso en pesos"*

### Pregunta 5 — ¿Hay algún día en que las ventas fueron menores a $200.000?

In [ ]:
# Primero armamos el resumen diario
resumen_diario = df.groupby('dia')['total'].sum().reset_index()
resumen_diario.columns = ['dia', 'total_dia']

# Luego filtramos
dias_bajos = resumen_diario[resumen_diario['total_dia'] < 200_000]

if len(dias_bajos) == 0:
    print("Ningún día tuvo ventas menores a $200.000")
else:
    print("Días con ventas bajas:")
    print(dias_bajos)

> `resumen_diario[resumen_diario['total_dia'] < 200_000]` es el filtro.  
> Léelo así: *"del resumen diario, dame solo las filas donde total_dia sea menor a 200.000"*  
> Es exactamente lo que hace el filtro de Excel, pero en una línea y aplicable a cualquier número de filas.

---
## Parte 4 — Primera gráfica

Los números están bien, pero una gráfica comunica más rápido.  
Pandas puede generar gráficas directamente desde un DataFrame.

In [ ]:
import matplotlib.pyplot as plt

# Ventas por día — ordenadas por día de la semana
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado']

ventas_por_dia = (
    df.groupby('dia')['total']
    .sum()
    .reindex(orden_dias)
)

# df.plot() — sin configurar nada extra
ventas_por_dia.plot(
    kind='bar',
    figsize=(10, 5),
    title='Ventas diarias — Cafetería Intenalco',
    color='#4f9cf9',
    edgecolor='white'
)

plt.xticks(rotation=0)
plt.ylabel('Total vendido ($)')
plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Participación por categoría
df.groupby('categoria')['total'].sum().plot(
    kind='pie',
    figsize=(6, 6),
    title='Participación en ventas por categoría',
    autopct='%1.0f%%',
    colors=['#4f9cf9', '#7c5af9'],
    ylabel=''
)

plt.tight_layout()
plt.show()

> 💬 **Prompt para la IA:** *"Modifica la gráfica de barras para que las barras estén ordenadas de mayor a menor venta, y cambia el color a verde"*

---
## Parte 5 — Reto de la célula ⚡

Trabaja con tu célula. Tienen **20 minutos**.

Respondan estas tres preguntas usando el código que aprendieron hoy.  
Pueden pedirle ayuda al asistente IA de Colab, pero deben entender lo que genera.

**Pregunta A**  
¿Cuál es el precio promedio por unidad de cada producto?  
*(Pista: `df.groupby('producto')['precio_und'].mean()`)*

**Pregunta B**  
¿En qué día se vendieron más unidades físicas (no en pesos)?  
*(Pista: cambia `'total'` por `'unidades'` en el groupby)*

**Pregunta C — nivel extra**  
Genera una tabla que muestre, por categoría Y por día, el total vendido.  
*(Pista: `df.groupby(['dia', 'categoria'])['total'].sum()`)* 

In [ ]:
# Pregunta A — precio promedio por producto


In [ ]:
# Pregunta B — día con más unidades vendidas


In [ ]:
# Pregunta C — tabla por categoría y día


---
## Resumen de lo aprendido hoy

| Operación | Para qué sirve | Equivalente en Excel |
|-----------|---------------|---------------------|
| `pd.read_csv(url)` | Cargar datos | Abrir archivo |
| `df.head()` | Ver primeras filas | Ctrl + Inicio |
| `df.info()` | Ver tipos y vacíos | Ninguno directo |
| `df.describe()` | Estadísticas básicas | Ninguno directo |
| `df['columna']` | Seleccionar columna | Clic en encabezado |
| `df['col'].sum()` | Sumar columna | =SUMA(rango) |
| `df[df['col'] > x]` | Filtrar filas | Filtro |
| `df.groupby('col')` | Agrupar | Tabla dinámica |
| `.sort_values()` | Ordenar | Ordenar A→Z |
| `.plot(kind='bar')` | Gráfica de barras | Insertar gráfico |

---

**Próxima sesión:** el mismo flujo pero con 180 filas y dos tablas que hay que cruzar.  
El BUSCARV va a quedar corto.